# Lab · Mixture of Experts (MoE)

**KEEPCODING · AI Engineering · Clase 2 · Módulo M11: Pretraining**

Los modelos frontera de 2026 (GPT-5, DeepSeek-V3, Mixtral, Qwen...) **no son densos: son MoE**.
En M10 cada bloque tenía **un MLP denso**; aquí vemos cómo los grandes lo sustituyen por **muchos expertos**
de los que solo se activan unos pocos por token. Acompaña a las slides *Mixture of Experts* de M11.

> **Cómo seguir este lab**
>
> - Solo **NumPy**, no hace falta GPU: construimos un MoE **conceptual** para *ver* las piezas (router, expertos, top-k).
> - Ejecuta en orden. No entrenamos un MoE real; el objetivo es entender qué pasa por dentro de Mixtral / DeepSeek-V3.

## ¿Por qué MoE? · la arquitectura de los modelos frontera

Hasta ahora hemos construido un Transformer **denso**: cada token pasa por **todos** los parametros en cada capa. Funciona, pero escalar a billones de parametros sin que la latencia se dispare es caro.

En 2026 casi todos los **frontier models** son **Mixture of Experts (MoE)** en lugar de densos. Algunos ejemplos publicos:

| Modelo | Parametros totales | Parametros activos por token |
|---|---|---|
| Mixtral 8x7B | ~47B | ~13B |
| DeepSeek-V3 | 671B | 37B |
| Qwen 3.5 | 397B | 17B |
| Gemma 4 26B A4B | 26B | ~3.8B |
| GPT-5 (rumored) | (no publico) | MoE casi seguro |

La idea: **mas parametros totales, pero menos computo por token**. Mas capacidad, mismo coste de inferencia.

> Importante: este notebook NO entrena un MoE. Solo construimos una version conceptual con NumPy para visualizar las piezas. El objetivo es que entiendas que pasa por dentro cuando lees un paper de Mixtral o DeepSeek-V3.

## 1 · Dense FFN vs MoE FFN · que cambia?

Recuerda la estructura del MLP / FFN denso de los bloques anteriores:

```
e' = e + W_down( ReLU( W_up( e ) ) )
```

Cada token e activa **todas las filas** de `W_up` y `W_down`. En GPT-3, esto son `116B` de parametros activados en cada paso por cada token. Caro.

En un MoE, el FFN se reemplaza por:

1. Un **conjunto de expertos** (cada uno es un FFN denso pequeno): `E_1, E_2, ..., E_N`.
2. Un **router** (una capa lineal pequena): mira al token y decide a que expertos enviarlo.
3. **Top-k routing**: solo los `k` expertos con mayor puntuacion (tipico `k=2`) procesan el token. Los demas se ignoran.
4. La salida es la **suma ponderada** de los expertos elegidos.

```
scores  = softmax( W_router @ e )       # un peso por experto
top_k   = indices con los k scores mas altos
e'      = e + sum_{i en top_k} scores[i] * E_i(e)
```

Resultado: para cada token solo se calculan `k` expertos de `N`. Si `N=8` y `k=2`, usas solo el `25%` del FFN, pero tienes acceso al `100%` del conocimiento que vive en los expertos.

## 2 · Toy MoE en NumPy

Vamos a montar un MoE minimo de un solo bloque con:

- `d_model = 16` (dimension del embedding)
- `d_ff    = 32` (dimension interna del FFN)
- `n_experts = 4`
- `top_k = 2`

Cada experto es un FFN denso de dos capas con GELU. El router es una matriz `d_model x n_experts`. Todo aleatorio: no entrenamos nada, solo demostramos el flujo.

In [ ]:
import numpy as np

np.random.seed(7)  # reproducibilidad: queremos los mismos numeros cada vez

D_MODEL   = 16   # dimension de los embeddings que entran y salen del bloque
D_FF      = 32   # dimension interna de cada experto (el "ancho" del FFN)
N_EXPERTS = 4    # cuantos expertos hay en este bloque MoE
TOP_K     = 2    # cuantos expertos procesan cada token

def gelu(x):
    # Aproximacion de GELU (la misma que usan GPT y muchos otros)
    return 0.5 * x * (1.0 + np.tanh(np.sqrt(2.0 / np.pi) * (x + 0.044715 * x**3)))


class Expert:
    """Un experto = un FFN denso de dos capas (igual que el MLP de un Transformer normal)."""
    def __init__(self, d_model: int, d_ff: int, name: str):
        self.W_up   = np.random.randn(d_model, d_ff) * 0.1
        self.W_down = np.random.randn(d_ff, d_model) * 0.1
        self.name   = name

    def __call__(self, x):
        # x shape: (d_model,) -> salida shape (d_model,)
        h = gelu(x @ self.W_up)
        return h @ self.W_down


# Creamos los expertos. En la vida real cada uno acabaria especializandose en
# patrones distintos: codigo, matematicas, idioma X, hechos sobre cocina, etc.
experts = [Expert(D_MODEL, D_FF, name=f"E{i}") for i in range(N_EXPERTS)]

# El router: una capa lineal pequena que mira el token y produce un score por experto.
W_router = np.random.randn(D_MODEL, N_EXPERTS) * 0.5

print(f"Tenemos {N_EXPERTS} expertos, top_k = {TOP_K}.")
print(f"Parametros por experto: {D_MODEL * D_FF * 2} ({D_MODEL}x{D_FF} + {D_FF}x{D_MODEL})")
print(f"Parametros totales del MoE (sin contar router): {N_EXPERTS * D_MODEL * D_FF * 2}")
print(f"Parametros activos por token:                   {TOP_K * D_MODEL * D_FF * 2}")


## 3 · El router en accion · top-k

Para cada token, el router calcula un score por experto y un softmax sobre esos scores. Despues nos quedamos con los `top_k` mas altos y renormalizamos para que sus pesos sumen 1.

> Esta renormalizacion es importante: si el router elige expertos con scores `(0.4, 0.3)`, queremos que la suma ponderada use pesos `(4/7, 3/7)`, no `(0.4, 0.3)`. Asi el bloque mantiene la magnitud de la senal estable.

In [ ]:
def softmax(z):
    # softmax numericamente estable (restar el max evita overflow)
    z = z - np.max(z)
    e = np.exp(z)
    return e / e.sum()


def route(token, W_router, top_k: int):
    """Decide a que expertos va este token y con que peso.

    Devuelve:
      - scores_full: softmax sobre los N expertos (para inspeccion)
      - top_idx:     indices de los top_k expertos elegidos
      - top_weights: pesos normalizados de esos expertos (suman 1)
    """
    logits      = token @ W_router                       # shape: (n_experts,)
    scores_full = softmax(logits)
    top_idx     = np.argsort(scores_full)[-top_k:][::-1] # los k mas altos, en orden descendente
    top_weights = scores_full[top_idx]
    top_weights = top_weights / top_weights.sum()        # renormalizar a suma 1
    return scores_full, top_idx, top_weights


# Probamos con un token aleatorio.
token_demo = np.random.randn(D_MODEL)
scores, idx, weights = route(token_demo, W_router, TOP_K)

print("Scores sobre TODOS los expertos:")
for i, s in enumerate(scores):
    marca = "  <-- elegido" if i in idx else ""
    print(f"  E{i}: {s:.4f}{marca}")

print(f"\nExpertos elegidos: {[f'E{i}' for i in idx]}")
print(f"Pesos renormalizados (suman 1): {weights.round(4).tolist()}")


## 4 · El bloque MoE completo

Ahora juntamos router + expertos. Por cada token:

1. El router decide los `top_k` expertos.
2. Cada experto elegido procesa el token.
3. Sumamos sus salidas ponderadas por los pesos del router.
4. Conexion residual: `e' = e + delta`.

Los expertos no elegidos **no se calculan** · este es el truco de eficiencia.

In [ ]:
def moe_block(token, W_router, experts, top_k: int, verbose: bool = False):
    scores, top_idx, top_weights = route(token, W_router, top_k)

    delta = np.zeros_like(token)
    expertos_llamados = []
    for w, i in zip(top_weights, top_idx):
        # Solo aqui llamamos al experto (los demas no consumen computo)
        delta += w * experts[i](token)
        expertos_llamados.append((experts[i].name, float(w)))

    out = token + delta  # conexion residual: igual que en el Transformer denso

    if verbose:
        print(f"Token enrutado a: {expertos_llamados}")
        print(f"Expertos no usados ({N_EXPERTS - top_k} de {N_EXPERTS}): "
              f"{[experts[j].name for j in range(N_EXPERTS) if j not in top_idx]}")
    return out


# Procesamos una secuencia de 6 tokens y mostramos a quien va cada uno.
np.random.seed(42)
secuencia = np.random.randn(6, D_MODEL)

print("=== Enrutado token a token ===")
salidas = []
for t, tok in enumerate(secuencia):
    print(f"\nToken {t}:")
    out = moe_block(tok, W_router, experts, TOP_K, verbose=True)
    salidas.append(out)
salidas = np.stack(salidas)

print(f"\nForma de la salida: {salidas.shape}  (igual que la entrada, como debe ser)")


## 5 · Visualizar el patron de enrutado

Una de las cosas mas interesantes de un MoE es ver **que experto le toca a cada token**. En modelos reales (entrenados), aparecen especializaciones · un experto se vuelve "el de codigo", otro "el de espanol", otro "el de matematicas".

Aqui los expertos son aleatorios, pero el grafico ya nos dice si el router esta dando uso equilibrado o esta favoreciendo a un experto.

In [ ]:
# Tabla: tokens (filas) x expertos (columnas). Cada celda es el peso (0 si no elegido).
N_TOKENS = 20
np.random.seed(123)
tokens_demo = np.random.randn(N_TOKENS, D_MODEL)

matriz_routing = np.zeros((N_TOKENS, N_EXPERTS))
for t, tok in enumerate(tokens_demo):
    _, idx, w = route(tok, W_router, TOP_K)
    for j, i_exp in enumerate(idx):
        matriz_routing[t, i_exp] = w[j]

# Imprimimos en consola (sin necesidad de matplotlib).
print("Patron de routing (peso por token y experto):")
print("            " + "  ".join(f"E{j}" for j in range(N_EXPERTS)))
for t in range(N_TOKENS):
    fila = "  ".join(f"{matriz_routing[t, j]:.2f}" if matriz_routing[t, j] > 0 else "  . " for j in range(N_EXPERTS))
    print(f"  tok {t:>2}:  {fila}")

# Carga total por experto (load balancing)
carga = matriz_routing.sum(axis=0)
print(f"\nCarga total por experto (suma de pesos): {carga.round(2).tolist()}")
print("Si los numeros estan muy desigualados, en un MoE real anadiriamos una")
print("perdida auxiliar de load balancing para forzar reparto equitativo.")


## 6 · Load balancing · el problema clave del entrenamiento

Si dejas al router a su aire, hay un riesgo: **colapso de expertos**. El router descubre pronto que `E0` da resultados decentes y empieza a enviarle todo. Los demas expertos no reciben gradiente, no aprenden, y acabas con un MoE caro que se comporta como un FFN denso peque?o.

Soluciones reales (Mixtral, DeepSeek-V3, Switch Transformer):

- **Auxiliary load-balancing loss**: a?adir un termino que penaliza desequilibrio en la carga de los expertos.
- **Capacity factor**: cada experto tiene un cupo maximo de tokens por batch. Si se llena, los siguientes tokens hacen *token dropping* o se reenrutan.
- **Noisy top-k routing**: anadir ruido gaussiano a los scores antes de elegir top-k (estimula exploracion).
- **DeepSeek-V3 / DeepSeekMoE**: arquitectura *fine-grained* con muchos expertos pequenos (256 expertos) + *shared experts* siempre activos.
- **Expert choice routing** (Google): invertir el problema, cada experto elige sus tokens en vez de cada token elegir sus expertos.

## 7 · Por que importa en 2026 · scaling laws

El argumento economico de MoE es brutal y por eso domina:

- **Compute por token**: si tu MoE tiene `1T` parametros totales pero solo `30B` activos, tu coste de inferencia es el de un denso `30B`.
- **Capacidad**: la calidad de un modelo escala mas con parametros totales que con activos. Mas parametros = mas hechos almacenables (recuerda (M10): "los hechos viven en los MLPs").
- **VRAM vs FLOPs**: MoE pide mucha memoria (todos los expertos viven en VRAM) pero pocos FLOPs por token. Encaja muy bien con clusters de GPUs modernas.
- **Inferencia distribuida**: cada GPU puede alojar un subconjunto de expertos · *expert parallelism*. Mixtral, DeepSeek-V3 y Qwen 3.5 explotan esto a escala.

Resumen practico para un AI engineer:

| Aspecto | Dense | MoE |
|---|---|---|
| Parametros activos por token | 100 % | k/N (tipico 12.5 %-25 %) |
| Latencia por token | Alta a gran escala | Baja para el mismo "tamano efectivo" |
| Coste de VRAM | Bajo | Alto (hay que cargar todos los expertos) |
| Estabilidad de entrenamiento | Sencilla | Compleja (load balancing) |
| Donde manda hoy | Modelos pequenos en edge | Frontier models (GPT-5, Claude 4.7, Gemini 3) |

### Recursos para profundizar

- Switch Transformer (Google, 2021) · el paper que popularizo MoE moderno.
- Mixtral 8x7B (Mistral, 2023) · primer MoE open weights de calidad GPT-4ish.
- DeepSeek-V3 (2024) · MoE con 671B totales / 37B activos, *shared experts*, MLA.
- Qwen 3.5 (Alibaba, 2025) · 397B totales / 17B activos.

**Punto clave**: cuando leas en 2026 que un modelo es "200B" parametros, mira siempre si esos son **totales o activos**. La diferencia entre `200B-total / 20B-activos` y `200B-denso` es un orden de magnitud en coste de inferencia.